In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#To split data into stratified train and test sets
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

#for XGBoost model data format
from sklearn.preprocessing import LabelEncoder 

# Import Model
import xgboost as xgb

#for hyperparameter tuning
from sklearn.model_selection import GridSearchCV

# Import Metrics
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import root_mean_squared_error as rmse
from scipy.special import huber
from sklearn.metrics import make_scorer, mean_pinball_loss

In [ ]:
#read in the cleaned data file
df = gen.load_training_data()
df2 = gen.load_testing_data()
#df = pd.read_csv("/home/katherine-schwind/Documents/full_cleaned_data_train.csv", low_memory=False)
#df2 = pd.read_csv("/home/katherine-schwind/Documents/full_cleaned_data_test.csv", low_memory=False)

In [3]:
#make two new data columns
df['NUM_CODES'] = df.iloc[:, 14:36].sum(axis = 1)
df['PROLONGED'] = [1 if los >= 30 else 0 for los in df["LENGTH_OF_STAY"]]

df.LENGTH_OF_STAY = df.LENGTH_OF_STAY.astype(int)

df2['NUM_CODES'] = df2.iloc[:, 14:36].sum(axis = 1)
df2['PROLONGED'] = [1 if los >= 30 else 0 for los in df2["LENGTH_OF_STAY"]]

df2.LENGTH_OF_STAY = df2.LENGTH_OF_STAY.astype(int)

In [4]:
cat_features = (
    df.columns[2:3].tolist() +
    df.columns[6:10].tolist() +
    df.columns[12:13].tolist() +
    df.columns[14:38].tolist() +
    df.columns[39:40].tolist() 
)
cat_features

['TYPE_OF_ADMISSION',
 'SEX_CODE',
 'RACE',
 'ETHNICITY',
 'ADMIT_WEEKDAY',
 'EMERGENCY_DEPT_FLAG',
 'CODE_1',
 'CODE_2',
 'CODE_3',
 'CODE_4',
 'CODE_5',
 'CODE_6',
 'CODE_7',
 'CODE_8',
 'CODE_9',
 'CODE_10',
 'CODE_11',
 'CODE_12',
 'CODE_13',
 'CODE_14',
 'CODE_15',
 'CODE_16',
 'CODE_17',
 'CODE_18',
 'CODE_19',
 'CODE_20',
 'CODE_21',
 'CODE_22',
 'PAT_RURAL',
 'PROVIDER_RURAL',
 'QUARTER']

In [6]:
#Define categorical features, and put into a data type that works with XGBoost
df[cat_features] = (
    df[cat_features]
    .fillna("Missing")
    .astype('category')
)

df2[cat_features] = (
    df2[cat_features]
    .fillna("Missing")
    .astype('category')
)

In [7]:
#Make list of all features to be used in the model, including categorical features and numerical features
features = cat_features.copy()
features.extend(['PAT_AGE','PAT_LATITUDE','PAT_LONGITUDE','PROVIDER_LATITUDE','PROVIDER_LONGITUDE','NUM_CODES','PAT2PROV_DISTANCE'])

features

['TYPE_OF_ADMISSION',
 'SEX_CODE',
 'RACE',
 'ETHNICITY',
 'ADMIT_WEEKDAY',
 'EMERGENCY_DEPT_FLAG',
 'CODE_1',
 'CODE_2',
 'CODE_3',
 'CODE_4',
 'CODE_5',
 'CODE_6',
 'CODE_7',
 'CODE_8',
 'CODE_9',
 'CODE_10',
 'CODE_11',
 'CODE_12',
 'CODE_13',
 'CODE_14',
 'CODE_15',
 'CODE_16',
 'CODE_17',
 'CODE_18',
 'CODE_19',
 'CODE_20',
 'CODE_21',
 'CODE_22',
 'PAT_RURAL',
 'PROVIDER_RURAL',
 'QUARTER',
 'PAT_AGE',
 'PAT_LATITUDE',
 'PAT_LONGITUDE',
 'PROVIDER_LATITUDE',
 'PROVIDER_LONGITUDE',
 'NUM_CODES',
 'PAT2PROV_DISTANCE']

In [8]:
#Split into training and test sets for final model evaluation
X_train = df[features]
y_train = df.LENGTH_OF_STAY

X_test = df2[features]
y_test = df2.LENGTH_OF_STAY

In [9]:
# Run the RMSE model on the full set and record results and feature importance
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

#encode target variable for XGBoost
le = LabelEncoder()
y_tr_encoded = le.fit_transform(y_tr)
y_val_encoded = le.fit_transform(y_val)
y_test_encoded = le.fit_transform(y_test)


def huber_loss(y_true, y_pred, delta=2.0):
    error = y_true - y_pred
    is_small = np.abs(error) <= delta

    squared = 0.5 * error**2
    linear = delta * (np.abs(error) - 0.5 * delta)

    return np.mean(np.where(is_small, squared, linear))

full_rmse_model = xgb.XGBRegressor(
    max_depth=4, #Tuned Hyperparameter 1
    learning_rate=0.075, #Tuned Hyperparameter 2
    min_child_weight=3, #Tuned Hyperparameter 3
    max_cat_threshold=16, #Tuned Hyperparameter 4
    n_estimators=4000,
    eval_metric="rmse",
    random_state=42,
    early_stopping_rounds=400,
    enable_categorical=True,
    objective = "reg:squarederror",
        )

full_rmse_model.fit(X_tr, y_tr_encoded, eval_set=[(X_val, y_val_encoded)], verbose=100)

# Find predictions and store metrics 
preds = full_rmse_model.predict(X_test)

rmse_pred = rmse(y_test_encoded,preds)
mae_pred = mae(y_test_encoded,preds)
quantile_pred = mean_pinball_loss(
    y_test_encoded,
    preds,
    alpha=0.987531903035467
)

huber_pred = huber_loss(y_test_encoded, preds, delta=2.0)


print(
f"RMSE | "
f"RMSE={rmse_pred:.3f}, "
f"MAE={mae_pred:.3f}, "
f"Huber=2.0={huber_pred:.3f}, "
f"Quantile={quantile_pred:.5f}"
)


[0]	validation_0-rmse:8.85324
[100]	validation_0-rmse:7.43369
[200]	validation_0-rmse:7.26748
[300]	validation_0-rmse:7.17562
[400]	validation_0-rmse:7.12468
[500]	validation_0-rmse:7.09863
[600]	validation_0-rmse:7.07902
[700]	validation_0-rmse:7.06003
[800]	validation_0-rmse:7.04481
[900]	validation_0-rmse:7.03379
[1000]	validation_0-rmse:7.02019
[1100]	validation_0-rmse:7.01282
[1200]	validation_0-rmse:7.00644
[1300]	validation_0-rmse:6.99562
[1400]	validation_0-rmse:6.99009
[1500]	validation_0-rmse:6.98620
[1600]	validation_0-rmse:6.98039
[1700]	validation_0-rmse:6.97938
[1800]	validation_0-rmse:6.97460
[1900]	validation_0-rmse:6.97161
[2000]	validation_0-rmse:6.96946
[2100]	validation_0-rmse:6.96402
[2200]	validation_0-rmse:6.96071
[2300]	validation_0-rmse:6.95966
[2400]	validation_0-rmse:6.95719
[2500]	validation_0-rmse:6.95597
[2600]	validation_0-rmse:6.95338
[2700]	validation_0-rmse:6.95043
[2800]	validation_0-rmse:6.94834
[2900]	validation_0-rmse:6.94768
[3000]	validation_0-rm